# Transaction Linking Experiment

**Goal**: Given a receipt/invoice image and a bank statement image, ask InternVL3 to identify
which debit transaction in the bank statement corresponds to the purchase on the receipt.

This is a **cross-document multi-image reasoning** task using concatenated pixel values.

In [ ]:
"""Cell 1: Imports and sys.path setup."""

import re
import sys
from pathlib import Path

import torch
import yaml
from IPython.display import display
from PIL import Image

# Add project root to path so we can import from models/, common/, etc.
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from common.pipeline_config import PipelineConfig  # noqa: E402
from common.simple_prompt_loader import SimplePromptLoader  # noqa: E402
from models.internvl3_image_preprocessor import InternVL3ImagePreprocessor  # noqa: E402
from models.registry import get_model  # noqa: E402

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

In [ ]:
"""Cell 2: Load experiment config from YAML."""

config_path = PROJECT_ROOT / "config" / "experiment_config.yml"
with config_path.open() as f:
    exp_config = yaml.safe_load(f)

# Extract key settings
model_type = exp_config["model"]["type"]
max_tiles = exp_config["model"]["max_tiles"]
max_new_tokens = exp_config["model"]["max_new_tokens"]
prompt_file = exp_config["prompts"]["file"]
prompt_key = exp_config["prompts"]["key"]
ground_truth = exp_config["ground_truth"]

# Data mode: "synthetic" or "real"
data_mode = exp_config["data"]["mode"]
data_dir = PROJECT_ROOT / exp_config["data"]["dir"]

print(f"Model: {model_type}, max_tiles: {max_tiles}, max_new_tokens: {max_new_tokens}")
print(f"Prompt: {prompt_file} -> {prompt_key}")
print(f"Data mode: {data_mode}")
print(f"Data dir: {data_dir}")

In [ ]:
from experiments.synthetic.generate_receipt import generate_receipt

# Load or generate receipt image
if data_mode == "real":
    receipt_path = data_dir / exp_config["data"]["real_receipt"]
    if not receipt_path.exists():
        raise FileNotFoundError(f"Real receipt not found: {receipt_path}")
    print(f"Using real receipt: {receipt_path}")
else:
    receipt_path = generate_receipt(data_dir / "synthetic_receipt.png")
    print(f"Generated synthetic receipt: {receipt_path}")

receipt_img = Image.open(receipt_path)
display(receipt_img)

In [ ]:
from experiments.synthetic.generate_bank_statement import generate_bank_statement

# Load or generate bank statement image
if data_mode == "real":
    statement_path = data_dir / exp_config["data"]["real_statement"]
    if not statement_path.exists():
        raise FileNotFoundError(f"Real bank statement not found: {statement_path}")
    print(f"Using real bank statement: {statement_path}")
else:
    statement_path = generate_bank_statement(data_dir / "synthetic_bank_statement.png")
    print(f"Generated synthetic bank statement: {statement_path}")

statement_img = Image.open(statement_path)
display(statement_img)

In [ ]:
"""Cell 5: Load InternVL3 via registry context manager."""

# Build a minimal PipelineConfig for the model loader
run_config_path = PROJECT_ROOT / "config" / "run_config.yml"
with run_config_path.open() as f:
    run_config = yaml.safe_load(f)

# Resolve model path: explicit model.path first, then default_paths lookup
model_path = run_config.get("model", {}).get("path")
if not model_path:
    model_path = (
        run_config.get("model_loading", {}).get("default_paths", {}).get(model_type)
    )
if not model_path:
    raise FileNotFoundError(
        f"No model path found for '{model_type}' in run_config.yml. "
        "Set model.path or model_loading.default_paths.{model_type}."
    )
model_path = Path(model_path)
print(f"Model path: {model_path}")

cfg = PipelineConfig(
    data_dir=data_dir,
    output_dir=data_dir,
    model_path=model_path,
    model_type=model_type,
    max_tiles=max_tiles,
    flash_attn=exp_config["model"]["flash_attn"],
    dtype=exp_config["model"]["dtype"],
    max_new_tokens=max_new_tokens,
)

# Get loader from registry
registration = get_model(model_type)
model_ctx = registration.loader(cfg)
model, tokenizer = model_ctx.__enter__()
print(f"Model loaded: {type(model).__name__}")
print(f"Device: {next(model.parameters()).device}")

In [ ]:
"""Cell 6: Preprocess both images — tile and concatenate pixel_values."""

preprocessor = InternVL3ImagePreprocessor(max_tiles=max_tiles)

# Load and tile each image
receipt_pv = preprocessor.load_image(str(receipt_path), model, max_num=max_tiles)
statement_pv = preprocessor.load_image(str(statement_path), model, max_num=max_tiles)

print(f"Receipt tiles: {receipt_pv.shape[0]}, Statement tiles: {statement_pv.shape[0]}")
print(f"Receipt pv shape: {receipt_pv.shape}")
print(f"Statement pv shape: {statement_pv.shape}")

# Concatenate pixel_values — each <image> tag maps to one image's patches
combined_pv = torch.cat([receipt_pv, statement_pv], dim=0)
num_patches = [receipt_pv.shape[0], statement_pv.shape[0]]

print(f"Combined shape: {combined_pv.shape}")
print(f"Num patches per image: {num_patches}")

In [ ]:
"""Cell 7: Load linking prompt from YAML."""

linking_prompt = SimplePromptLoader.load_prompt(prompt_file, prompt_key)
print(f"Prompt key: {prompt_key}")
print(f"Prompt length: {len(linking_prompt)} chars")
print("---")
print(linking_prompt)

In [ ]:
# Run multi-image inference with model.chat()
# InternVL3.5 .chat() expects generation_config as a dict, not GenerationConfig
generation_config = {
    "max_new_tokens": max_new_tokens,
    "do_sample": False,
    "num_beams": 1,
}

# Two <image> tags — one per source image
question = f"<image>\n<image>\n{linking_prompt}"

print("Running inference...")
response = model.chat(
    tokenizer,
    combined_pv,
    question,
    generation_config,
    num_patches_list=num_patches,
)

print("=" * 60)
print("MODEL RESPONSE")
print("=" * 60)
print(response)

In [ ]:
"""Cell 9: Parse response — regex key-value extraction."""


def parse_kv_response(text: str) -> dict[str, str]:
    """Extract KEY: VALUE pairs from model response."""
    pattern = re.compile(r"^([A-Z_]+):\s*(.+)$", re.MULTILINE)
    return {m.group(1): m.group(2).strip() for m in pattern.finditer(text)}


parsed = parse_kv_response(response)
print(f"Extracted {len(parsed)} fields:")
for key, value in parsed.items():
    print(f"  {key}: {value}")

In [ ]:
"""Cell 10: Validate against ground truth."""

# Ground truth from experiment config
gt = ground_truth

# Normalise keys to upper-case for comparison
gt_normalised = {k.upper(): v for k, v in gt.items()}

print("Validation Results")
print("=" * 60)

all_pass = True
for field, expected in gt_normalised.items():
    actual = parsed.get(field, "MISSING")
    match = actual.upper() == expected.upper()
    status = "PASS" if match else "FAIL"
    if not match:
        all_pass = False
    print(f"  [{status}] {field}: expected='{expected}' got='{actual}'")

print("=" * 60)
if all_pass:
    print("ALL FIELDS MATCH — experiment passed.")
else:
    print("SOME FIELDS DID NOT MATCH — review above.")

In [ ]:
import gc

# Cleanup — free GPU memory
try:
    model_ctx.__exit__(None, None, None)
except Exception:
    pass

# Belt-and-braces cleanup
del combined_pv, receipt_pv, statement_pv
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print("Cleanup complete.")